In [1]:
# import pandas 
import pandas as pd
# import numpy
import numpy as np
# import matplotlib
from matplotlib import pyplot as plt
# make plots appear and be stored within the notebook
%matplotlib inline

In [3]:
df = pd.read_csv(
    '/Users/ruochenliu/Documents/ESE5023/Assignment_2/ibtracs.ALL.list.v04r00.csv',
    usecols=range(17),
    skiprows=[1, 2],
    parse_dates=['ISO_TIME'],
    na_values=['NOT_NAMED', 'NAME']
)

df.head()

/var/folders/lt/y9v5bfd53yd3nf8jpwgwh6lc0000gn/T/ipykernel_84234/4155827071.py:1: DtypeWarning: Columns (0: NAME) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(


,SID,SEASON,NUMBER,BASIN,SUBBASIN,NAME,ISO_TIME,NATURE,LAT,LON,WMO_WIND,WMO_PRES,WMO_AGENCY,TRACK_TYPE,DIST2LAND,LANDFALL,IFLAG
0,1842298N11080,1842,1,NI,BB,NaN,1842-10-25 06:00:00,NR,10.8709,79.8265,,,,main,0,0,__________P___
1,1842298N11080,1842,1,NI,BB,NaN,1842-10-25 09:00:00,NR,10.8431,79.3524,,,,main,0,0,__________P___
2,1842298N11080,1842,1,NI,BB,NaN,1842-10-25 12:00:00,NR,10.8188,78.8772,,,,main,0,0,__________P___
3,1842298N11080,1842,1,NI,BB,NaN,1842-10-25 15:00:00,NR,10.8000,78.4000,,,,main,0,0,__________O___
4,1842298N11080,1842,1,NI,AS,NaN,1842-10-25 18:00:00,NR,10.7884,77.9194,,,,main,0,0,__________P___


In [ ]:
df['WMO_WIND'] = pd.to_numeric(df['WMO_WIND'], errors='coerce')#先将风速列转换为数字，可以进行大小对比，并且把缺失值都补全为NaN

storm_winds = df.groupby('SID').agg({'NAME': 'first','WMO_WIND': 'max'}).reset_index()#排序，选出最大值

storm_winds = storm_winds.dropna(subset=['NAME', 'WMO_WIND'])#这里先筛选没有值的风暴
storm_winds = storm_winds.sort_values('WMO_WIND', ascending=False)

In [ ]:
#3.1
storm_winds[['NAME', 'WMO_WIND']].head(10)

In [ ]:
#3.2
storm_winds['WMO_WIND'].head(20)

In [ ]:
#3.3
df['BASIN'].value_counts()

In [ ]:
#3.4
df.plot.hexbin(x='LON', y='LAT', gridsize=40, figsize=(6, 4), title='Tropical Cyclone Data Points')

In [ ]:
#3.5
mangkhut = df[(df['NAME'] == 'MANGKHUT') & (df['ISO_TIME'].dt.year == 2018)] #选出该风暴

mangkhut.plot(x='LON', y='LAT', figsize=(6, 4), title='Typhoon Mangkhut (2018)')#对该风暴做图

In [ ]:
#3.6
df_wp_ep = df[(df['ISO_TIME'].dt.year >= 1970) & (df['BASIN'].isin(['WP', 'EP']))].copy()
df_wp_ep.info()

In [ ]:
#3.7
daily_counts = df_wp_ep.set_index('ISO_TIME').resample('D').size()
daily_counts.plot(figsize=(10, 4), xlabel='Date', ylabel='Data points per day', title='Daily IBTrACS Data Points')

In [ ]:
#3.8
climatology = daily_counts.groupby(daily_counts.index.dayofyear).mean()
print(climatology)

In [ ]:
#3.9
average_by_day = daily_counts.groupby(daily_counts.index.dayofyear).mean()# 先查看每个日历日对应的平均数据点数

daily_anomaly = daily_counts - daily_counts.index.dayofyear.map(average_by_day)#计算异常，每一天的异常值 = 当天数据点数 - 该日历日的平均数
display(daily_anomaly)

In [ ]:
#3.10
annual_anomaly = daily_anomaly.resample('YE').sum() #从每日异常值重新计算年度结果，用日期索引
annual_anomaly.index = annual_anomaly.index.year #把日期索引转换成年份索引

annual_anomaly.plot(kind='bar', figsize=(12, 4), xlabel='Year', ylabel='Annual anomaly', title='Annual Tropical Cyclone Activity Anomaly')